In [16]:
import torch
from torch.utils.data import DataLoader
from src.load import MasterDataset, _worker_init_fn
from src.transforms import make_transform
from src.model import GravInvNet

ds = MasterDataset("data/master.h5")
ds.transform = make_transform(ds.shape_cells)

def collate(batch):
    xs, ys, masks = [], [], []
    for x, y, mask, _ in batch:
        xs.append(x)
        ys.append(y)
        masks.append(mask)
    return (
        torch.stack(xs, dim=0),     # (B,1,H,W)
        torch.stack(ys, dim=0),     # (B,Z,H,W)
        torch.stack(masks, dim=0),  # (B,Z,H,W)
    )
loader = DataLoader(
    ds,
    batch_size=4,
    shuffle=True,
    num_workers=2,
    worker_init_fn=_worker_init_fn,
    collate_fn=collate,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
net = GravInvNet().to(device)
optimizer = torch.optim.SGD(net.parameters(), lr=0.1, momentum=0.9)
net.train()
gz, tgt, mask = next(iter(loader))
gz, tgt, mask = gz.to(device), tgt.to(device), mask.to(device)
optimizer.zero_grad()
pred = net(gz)                     # (B,Z,H,W)
loss = ((pred - tgt)[mask] ** 2).mean()
loss.backward()
optimizer.step()
print(f"Loss: {loss.item():.6f}")


Using device: cuda
Loss: 0.100392
